# SEN12FLOOD Flood Proxy Labels (Water + Severity)

This notebook builds proxy labels for two tasks:
- **Segmentation**: water masks from Sentinel-2 (NDWI/mNDWI) with Sentinel-1 fallback.
- **Severity classification**: bins 0–3 from water extent + temporal behavior.

Main outputs:
- `proxy_outputs/masks_raw/`
- `proxy_outputs/masks_temporal/`
- `proxy_outputs/proxy_labels_raw.csv`
- `proxy_outputs/proxy_labels_temporal.csv`
- `proxy_outputs/proxy_labels_severity.csv`

In [ ]:
import os
import re
import glob
import json
import random
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _version_tuple(v):
    nums = re.findall(r'\d+', v)[:3]
    nums = [int(x) for x in nums] + [0, 0, 0]
    return tuple(nums[:3])

# rasterio can break with very new NumPy versions
if _version_tuple(np.__version__) >= (2, 1, 0):
    print(f'Detected NumPy {np.__version__}. Installing NumPy 1.26.4 for rasterio compatibility...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy==1.26.4'])
    raise RuntimeError('NumPy was changed. Please restart the kernel, then rerun Cell 2.')

try:
    import rasterio
except Exception as e:
    print('Installing rasterio...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'rasterio'])
    raise RuntimeError('Rasterio was installed/updated. Please restart the kernel, then rerun Cell 2.') from e

from rasterio.enums import Resampling

# repo-relative paths
data_dir = Path('./SEN12FLOOD')
out_dir = Path('./proxy_outputs')

raw_mask_dir = out_dir / 'masks_raw'
smooth_mask_dir = out_dir / 'masks_temporal'
raw_mask_dir.mkdir(parents=True, exist_ok=True)
smooth_mask_dir.mkdir(parents=True, exist_ok=True)

print('NumPy version:', np.__version__)
print('Data dir exists:', data_dir.exists())
print('Output dir:', out_dir)


In [ ]:
# helper funcs for scanning files and parsing names

def is_s2_band_file(path: Path):
    n = path.name
    return ('_B' in n) and n.lower().endswith('.tif')

def band_code_from_name(name: str):
    m = re.search(r'_B(\d{2})\.tif$', name)
    return f'B{m.group(1)}' if m else None

def scene_id_without_band(name: str):
    return re.sub(r'_B\d{2}\.tif$', '', name)

def parse_date_from_text(text: str):
    # supports YYYYMMDD and YYYY-MM-DD
    m1 = re.search(r'(20\d{2})(\d{2})(\d{2})', text)
    if m1:
        return f'{m1.group(1)}-{m1.group(2)}-{m1.group(3)}'
    m2 = re.search(r'(20\d{2}-\d{2}-\d{2})', text)
    if m2:
        return m2.group(1)
    return None

def build_s2_scene_table(root: Path):
    tif_paths = [Path(p) for p in glob.glob(str(root / '**/*.tif'), recursive=True)]
    s2_paths = [p for p in tif_paths if is_s2_band_file(p)]

    scene_map = {}
    for p in s2_paths:
        band = band_code_from_name(p.name)
        if band is None:
            continue
        scene_id = scene_id_without_band(p.name)
        region = p.parent.name
        key = (region, scene_id)
        if key not in scene_map:
            scene_map[key] = {'region': region, 'scene_id': scene_id, 'date': parse_date_from_text(scene_id)}
        scene_map[key][band] = str(p)

    records = []
    for (_, _), row in scene_map.items():
        records.append(row)

    df = pd.DataFrame(records)
    if len(df) == 0:
        return df

    # needed for NDWI + mNDWI
    needed = ['B03', 'B08', 'B11']
    for b in needed:
        if b not in df.columns:
            df[b] = np.nan
    df['has_required_s2_bands'] = df[needed].notna().all(axis=1)
    return df

def build_s1_file_table(root: Path):
    tif_paths = [Path(p) for p in glob.glob(str(root / '**/*.tif'), recursive=True)]

    def maybe_s1(p):
        n = p.name.upper()
        return ('S1' in n) or ('S1' in str(p).upper())

    s1_paths = [p for p in tif_paths if maybe_s1(p)]

    records = []
    for p in s1_paths:
        region = p.parent.name
        sid = p.stem
        records.append({
            'region': region,
            'scene_id': sid,
            'date': parse_date_from_text(sid),
            's1_path': str(p)
        })

    return pd.DataFrame(records)

In [ ]:
# build scene tables

df_s2 = build_s2_scene_table(data_dir)
df_s1 = build_s1_file_table(data_dir)

print('S2 grouped scenes:', len(df_s2))
if len(df_s2):
    print('S2 with required bands:', int(df_s2['has_required_s2_bands'].sum()))
    display(df_s2.head(3))

print('S1 files:', len(df_s1))
if len(df_s1):
    display(df_s1.head(3))

In [ ]:
# water-mask methods

def read_band(path, out_shape=None):
    with rasterio.open(path) as src:
        if out_shape is None:
            arr = src.read(1).astype(np.float32)
        else:
            arr = src.read(1, out_shape=out_shape, resampling=Resampling.bilinear).astype(np.float32)
    return arr

def safe_norm_diff(a, b, eps=1e-6):
    return (a - b) / (a + b + eps)

def optical_quality_mask(green, nir, swir1):
    valid = np.isfinite(green) & np.isfinite(nir) & np.isfinite(swir1)
    valid &= (green > 0) & (nir > 0) & (swir1 > 0)
    return valid

def optical_frame_quality_score(valid_mask):
    return float(valid_mask.mean())

def median_filter_u8(img, ksize=5):
    pad = ksize // 2
    padded = np.pad(img, ((pad, pad), (pad, pad)), mode='edge')
    windows = []
    for i in range(ksize):
        for j in range(ksize):
            windows.append(padded[i:i + img.shape[0], j:j + img.shape[1]])
    stack = np.stack(windows, axis=0)
    return np.median(stack, axis=0).astype(np.uint8)

def majority_filter(mask_bool, ksize=3, votes_required=None):
    mask = mask_bool.astype(np.uint8)
    pad = ksize // 2
    padded = np.pad(mask, ((pad, pad), (pad, pad)), mode='edge')
    s = np.zeros_like(mask, dtype=np.uint16)
    for i in range(ksize):
        for j in range(ksize):
            s += padded[i:i + mask.shape[0], j:j + mask.shape[1]]
    if votes_required is None:
        votes_required = (ksize * ksize) // 2 + 1
    return s >= votes_required

def cleanup_binary(mask_bool):
    x = majority_filter(mask_bool, ksize=3, votes_required=5)
    x = majority_filter(x, ksize=3, votes_required=5)
    return x

def otsu_threshold_u8(img_u8):
    hist = np.bincount(img_u8.ravel(), minlength=256).astype(np.float64)
    prob = hist / (hist.sum() + 1e-12)
    omega = np.cumsum(prob)
    mu = np.cumsum(prob * np.arange(256))
    mu_t = mu[-1]
    sigma_b2 = (mu_t * omega - mu) ** 2 / (omega * (1.0 - omega) + 1e-12)
    return int(np.nanargmax(sigma_b2))

def water_mask_s2_ndwi_mndwi(b03_path, b08_path, b11_path,
                           ndwi_thr=0.10, mndwi_thr=0.12,
                           valid_ratio_min=0.60):
    green = read_band(b03_path)
    h, w = green.shape
    nir = read_band(b08_path, out_shape=(1, h, w))
    swir1 = read_band(b11_path, out_shape=(1, h, w))

    qmask = optical_quality_mask(green, nir, swir1)
    qscore = optical_frame_quality_score(qmask)
    optical_ok = qscore >= valid_ratio_min

    ndwi = safe_norm_diff(green, nir)
    mndwi = safe_norm_diff(green, swir1)

    water = (ndwi > ndwi_thr) & (mndwi > mndwi_thr) & qmask
    water = cleanup_binary(water)

    return water, {'source': 'S2', 'optical_ok': bool(optical_ok), 'quality_score': qscore}

def water_mask_s1_adaptive(s1_path):
    arr = read_band(s1_path)
    arr = np.nan_to_num(arr, nan=np.nanmedian(arr))

    # log-compress + normalize
    arr = np.log1p(np.maximum(arr, 0))
    p2, p98 = np.percentile(arr, [2, 98])
    arr = np.clip(arr, p2, p98)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-6)
    img_u8 = (arr * 255).astype(np.uint8)

    # reduce speckle a bit
    den = median_filter_u8(img_u8, ksize=5)

    # in SAR, darker often corresponds to water
    thr = otsu_threshold_u8(den)
    water = den < thr

    water = cleanup_binary(water)

    return water, {'source': 'S1', 'otsu_thr': float(thr)}

In [ ]:
# generate raw proxy masks
records = []

# prefer S2 when possible; fallback to S1 per region
s2_lookup = {}
if len(df_s2):
    for _, r in df_s2.iterrows():
        s2_lookup[(r['region'], r['scene_id'])] = r

s1_by_region = {}
if len(df_s1):
    for _, r in df_s1.iterrows():
        s1_by_region.setdefault(r['region'], []).append(r['s1_path'])

if len(df_s2) == 0 and len(df_s1) == 0:
    raise RuntimeError('No S1/S2 scenes found. Check data_dir path and SEN12FLOOD file naming.')

for idx, row in df_s2.iterrows() if len(df_s2) else []:
    region = row['region']
    scene_id = row['scene_id']
    date = row.get('date', None)

    mask = None
    method = None
    quality_score = np.nan

    if bool(row.get('has_required_s2_bands', False)):
        try:
            m_s2, meta_s2 = water_mask_s2_ndwi_mndwi(row['B03'], row['B08'], row['B11'])
            if meta_s2['optical_ok']:
                mask = m_s2
                method = 'S2_NDWI_mNDWI'
                quality_score = meta_s2['quality_score']
        except Exception:
            pass

    if mask is None and region in s1_by_region and len(s1_by_region[region]) > 0:
        try:
            s1_path = s1_by_region[region][0]
            m_s1, meta_s1 = water_mask_s1_adaptive(s1_path)
            mask = m_s1
            method = 'S1_ADAPTIVE'
        except Exception:
            pass

    if mask is None:
        continue

    water_fraction = float(mask.mean())
    mask_file = raw_mask_dir / f'{region}__{scene_id}.npy'
    np.save(mask_file, mask.astype(np.uint8))

    records.append({
        'region': region,
        'scene_id': scene_id,
        'date': date,
        'method': method,
        'quality_score': quality_score,
        'water_fraction': water_fraction,
        'raw_mask_path': str(mask_file)
    })

df_raw = pd.DataFrame(records)
df_raw.to_csv(out_dir / 'proxy_labels_raw.csv', index=False)
print('Raw proxy rows:', len(df_raw))
display(df_raw.head(5))

In [ ]:
# temporal smoothing with 3-frame majority vote

def resize_nearest(mask, target_shape):
    """Resize 2D mask to target_shape (H, W) with nearest-neighbor indexing."""
    th, tw = target_shape
    h, w = mask.shape
    if (h, w) == (th, tw):
        return mask
    y_idx = np.linspace(0, h - 1, th).round().astype(int)
    x_idx = np.linspace(0, w - 1, tw).round().astype(int)
    return mask[np.ix_(y_idx, x_idx)]

def canonical_shape(masks):
    """Pick most common shape; tie-break with largest area."""
    shape_counts = {}
    for m in masks:
        shape_counts[m.shape] = shape_counts.get(m.shape, 0) + 1
    best = sorted(shape_counts.items(), key=lambda kv: (kv[1], kv[0][0] * kv[0][1]), reverse=True)[0][0]
    return best

def temporal_majority_3(masks):
    if len(masks) == 0:
        return []

    target_shape = canonical_shape(masks)
    aligned = [resize_nearest((m > 0).astype(np.uint8), target_shape) for m in masks]

    out = []
    n = len(aligned)
    for i in range(n):
        left = max(0, i - 1)
        right = min(n, i + 2)
        stack = np.stack(aligned[left:right], axis=0)
        vote = (stack.sum(axis=0) >= ((stack.shape[0] // 2) + 1)).astype(np.uint8)
        out.append(vote)
    return out

if len(df_raw) == 0:
    raise RuntimeError('No raw proxy masks available for temporal smoothing.')

df_tmp = df_raw.copy()
df_tmp['date_sort'] = pd.to_datetime(df_tmp['date'], errors='coerce')
df_tmp['date_sort'] = df_tmp['date_sort'].fillna(pd.Timestamp('1970-01-01'))

smooth_records = []
for region, g in df_tmp.sort_values(['region', 'date_sort', 'scene_id']).groupby('region'):
    paths = g['raw_mask_path'].tolist()
    masks = [np.load(p) for p in paths]
    smoothed = temporal_majority_3(masks)

    for (_, row), m in zip(g.iterrows(), smoothed):
        out_path = smooth_mask_dir / f"{row['region']}__{row['scene_id']}.npy"
        np.save(out_path, m.astype(np.uint8))
        smooth_records.append({
            'region': row['region'],
            'scene_id': row['scene_id'],
            'date': row['date'],
            'method': row['method'],
            'quality_score': row['quality_score'],
            'water_fraction_raw': row['water_fraction'],
            'water_fraction_temporal': float((m > 0).mean()),
            'raw_mask_path': row['raw_mask_path'],
            'temporal_mask_path': str(out_path),
            'temporal_mask_shape': str(m.shape)
        })

df_temp = pd.DataFrame(smooth_records)
df_temp.to_csv(out_dir / 'proxy_labels_temporal.csv', index=False)
print('Temporal rows:', len(df_temp))
display(df_temp.head(5))

In [ ]:
# compute severity score + bins
if len(df_temp) == 0:
    raise RuntimeError('Temporal proxy table is empty.')

df_sev = df_temp.copy()

# region-level persistence proxy
region_mean = df_sev.groupby('region')['water_fraction_temporal'].transform('mean')
df_sev['persistence_score'] = region_mean

# positive deviation from region baseline
df_sev['delta_score'] = (df_sev['water_fraction_temporal'] - region_mean).clip(lower=0)

# weighted severity score
df_sev['severity_score'] = (
    0.60 * df_sev['water_fraction_temporal'] +
    0.25 * df_sev['persistence_score'] +
    0.15 * df_sev['delta_score']
)

# fixed thresholds for now
thr = {
    'mild': 0.10,
    'moderate': 0.25,
    'severe': 0.45
}

def to_bin(s):
    if s < thr['mild']:
        return 0
    if s < thr['moderate']:
        return 1
    if s < thr['severe']:
        return 2
    return 3

df_sev['severity_bin'] = df_sev['severity_score'].apply(to_bin).astype(int)

df_sev.to_csv(out_dir / 'proxy_labels_severity.csv', index=False)
print(df_sev['severity_bin'].value_counts().sort_index())
display(df_sev.head(10))

In [ ]:
# small stratified hand-check set
n_per_bin = 8
sample_rows = []
for b in sorted(df_sev['severity_bin'].unique()):
    sub = df_sev[df_sev['severity_bin'] == b]
    k = min(n_per_bin, len(sub))
    if k > 0:
        sample_rows.append(sub.sample(k, random_state=42))

df_handcheck = pd.concat(sample_rows, ignore_index=True) if len(sample_rows) else pd.DataFrame()
df_handcheck.to_csv(out_dir / 'handcheck_subset.csv', index=False)
print('Hand-check subset size:', len(df_handcheck))
display(df_handcheck[['region', 'scene_id', 'method', 'water_fraction_temporal', 'severity_score', 'severity_bin']].head(20))

# quick look at first few masks
show_n = min(6, len(df_handcheck))
if show_n > 0:
    fig, axes = plt.subplots(1, show_n, figsize=(4*show_n, 4))
    if show_n == 1:
        axes = [axes]

    for i in range(show_n):
        r = df_handcheck.iloc[i]
        m = np.load(r['temporal_mask_path'])
        axes[i].imshow(m, cmap='Blues')
        axes[i].set_title(f"bin={int(r['severity_bin'])}\n{r['region']}")
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

## Report Plots and Tables



In [ ]:
# report tables + plots
from pathlib import Path

fig_dir = out_dir / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

# load from csv if not already in memory
if 'df_sev' not in globals() or df_sev is None or len(df_sev) == 0:
    sev_csv = out_dir / 'proxy_labels_severity.csv'
    if not sev_csv.exists():
        raise RuntimeError('proxy_labels_severity.csv not found. Run previous pipeline cells first.')
    df_sev = pd.read_csv(sev_csv)

if 'df_temp' not in globals() or df_temp is None or len(df_temp) == 0:
    temp_csv = out_dir / 'proxy_labels_temporal.csv'
    if temp_csv.exists():
        df_temp = pd.read_csv(temp_csv)

required_cols = {'region', 'severity_bin', 'severity_score', 'water_fraction_temporal'}
missing = required_cols - set(df_sev.columns)
if missing:
    raise RuntimeError(f'Missing required columns in df_sev: {missing}')

# 1) bin counts by region
region_bin = (
    df_sev.groupby(['region', 'severity_bin'])
          .size()
          .unstack(fill_value=0)
          .sort_index(axis=1)
          .sort_values(by=list(df_sev['severity_bin'].unique()), ascending=False, ignore_index=False)
)
region_bin.to_csv(out_dir / 'severity_bin_distribution_by_region.csv')

# 2) global bin counts
global_bin = (
    df_sev['severity_bin']
    .value_counts()
    .sort_index()
    .rename_axis('severity_bin')
    .reset_index(name='count')
)
global_bin['percent'] = 100.0 * global_bin['count'] / global_bin['count'].sum()
global_bin.to_csv(out_dir / 'severity_bin_distribution_global.csv', index=False)

# 3) region summary stats
region_summary = (
    df_sev.groupby('region')
          .agg(
              n_samples=('severity_bin', 'size'),
              mean_water_fraction=('water_fraction_temporal', 'mean'),
              mean_severity_score=('severity_score', 'mean'),
              median_severity_score=('severity_score', 'median')
          )
          .sort_values('mean_severity_score', ascending=False)
)
region_summary.to_csv(out_dir / 'severity_region_summary.csv')

# plot A: stacked bins by region
plot_df = region_bin.copy()
plot_df = plot_df.sort_values(by=plot_df.columns.tolist(), ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
plot_df.plot(kind='bar', stacked=True, ax=ax)
ax.set_title('Severity Bin Distribution by Region')
ax.set_xlabel('Region')
ax.set_ylabel('Count')
ax.legend(title='Severity Bin', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
fig.savefig(fig_dir / 'severity_bins_by_region_stacked.png', dpi=200)
plt.show()

# plot B: global distribution
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(global_bin['severity_bin'].astype(str), global_bin['count'])
ax.set_title('Global Severity Bin Distribution')
ax.set_xlabel('Severity Bin')
ax.set_ylabel('Count')
for i, row in global_bin.iterrows():
    ax.text(i, row['count'], f"{row['percent']:.1f}%", ha='center', va='bottom', fontsize=9)
plt.tight_layout()
fig.savefig(fig_dir / 'severity_bins_global.png', dpi=200)
plt.show()

# plot C: severity score histogram
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df_sev['severity_score'].dropna(), bins=30)
ax.set_title('Severity Score Histogram')
ax.set_xlabel('Severity Score')
ax.set_ylabel('Frequency')
plt.tight_layout()
fig.savefig(fig_dir / 'severity_score_histogram.png', dpi=200)
plt.show()

print('Saved report tables:')
print('-', out_dir / 'severity_bin_distribution_by_region.csv')
print('-', out_dir / 'severity_bin_distribution_global.csv')
print('-', out_dir / 'severity_region_summary.csv')

print('\nSaved report figures:')
print('-', fig_dir / 'severity_bins_by_region_stacked.png')
print('-', fig_dir / 'severity_bins_global.png')
print('-', fig_dir / 'severity_score_histogram.png')

display(global_bin)
display(region_summary.head(10))

## Final Submission Helper Cells


In [ ]:
# optional calibration + manual check + packaging
import zipfile
from datetime import datetime

if 'df_sev' not in globals() or df_sev is None or len(df_sev) == 0:
    sev_csv = out_dir / 'proxy_labels_severity.csv'
    if not sev_csv.exists():
        raise RuntimeError('proxy_labels_severity.csv not found. Run pipeline cells first.')
    df_sev = pd.read_csv(sev_csv)

# A) optional quantile calibration
use_quantile_calibration = False

df_cal = df_sev.copy()
if use_quantile_calibration:
    q1, q2, q3 = df_cal['severity_score'].quantile([0.25, 0.50, 0.75]).values
    thr_auto = {'mild': float(q1), 'moderate': float(q2), 'severe': float(q3)}

    def to_bin_cal(s):
        if s < thr_auto['mild']:
            return 0
        if s < thr_auto['moderate']:
            return 1
        if s < thr_auto['severe']:
            return 2
        return 3

    df_cal['severity_bin_calibrated'] = df_cal['severity_score'].apply(to_bin_cal).astype(int)
    df_cal.to_csv(out_dir / 'proxy_labels_severity_calibrated.csv', index=False)
    print('Quantile calibration applied:', thr_auto)
    print('Saved:', out_dir / 'proxy_labels_severity_calibrated.csv')
else:
    print('Quantile calibration OFF (using original severity bins).')

# B) manual validation template
template_path = out_dir / 'handcheck_template_to_fill.csv'
subset_path = out_dir / 'handcheck_subset.csv'

if template_path.exists():
    df_hand = pd.read_csv(template_path).copy()
elif subset_path.exists():
    df_hand = pd.read_csv(subset_path).copy()
else:
    raise RuntimeError('No handcheck CSV found. Run hand-check generation cell first.')

if 'manual_severity_bin' not in df_hand.columns:
    df_hand['manual_severity_bin'] = np.nan
if 'manual_notes' not in df_hand.columns:
    df_hand['manual_notes'] = ''

# clean manual bin values
df_hand['manual_severity_bin'] = pd.to_numeric(df_hand['manual_severity_bin'], errors='coerce')
valid_bin_mask = df_hand['manual_severity_bin'].isin([0, 1, 2, 3])
df_hand.loc[~valid_bin_mask, 'manual_severity_bin'] = np.nan

df_hand.to_csv(template_path, index=False)
print('Manual template path:', template_path)
print('Accepted manual bins: 0,1,2,3 (others treated as missing).')

filled = df_hand['manual_severity_bin'].notna().sum()
if filled > 0:
    valid = df_hand[df_hand['manual_severity_bin'].notna()].copy()
    valid['manual_severity_bin'] = valid['manual_severity_bin'].astype(int)
    valid['model_bin'] = pd.to_numeric(valid['severity_bin'], errors='coerce').fillna(-1).astype(int)
    valid['agree'] = (valid['manual_severity_bin'] == valid['model_bin']).astype(int)
    agreement = float(valid['agree'].mean())

    per_bin = (
        valid.groupby('manual_severity_bin')['agree']
        .agg(['count', 'mean'])
        .rename(columns={'mean': 'agreement'})
    )
    per_bin.to_csv(out_dir / 'manual_validation_per_bin.csv')
    valid.to_csv(out_dir / 'handcheck_with_agreement.csv', index=False)

    print(f'Manual validation samples with labels: {len(valid)}')
    print(f'Overall agreement: {agreement:.3f}')
    display(per_bin)
else:
    print('No valid manual labels found yet in `manual_severity_bin`.')

# C) short executive summary text
global_dist = (
    df_sev['severity_bin']
    .value_counts()
    .sort_index()
    .rename_axis('severity_bin')
    .reset_index(name='count')
)
global_dist['percent'] = 100.0 * global_dist['count'] / global_dist['count'].sum()

top_regions = (
    df_sev.groupby('region')['severity_score']
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

summary_lines = []
summary_lines.append('SEN12FLOOD Proxy Label Pipeline - Executive Summary')
summary_lines.append(f'Generated at: {datetime.now().isoformat(timespec="seconds")}')
summary_lines.append('')
summary_lines.append(f'Total samples with severity labels: {len(df_sev)}')
summary_lines.append('Global severity distribution:')
for _, row in global_dist.iterrows():
    summary_lines.append(f"  Bin {int(row['severity_bin'])}: {int(row['count'])} ({row['percent']:.2f}%)")
summary_lines.append('')
summary_lines.append('Top regions by mean severity score:')
for region_name, score in top_regions.items():
    summary_lines.append(f'  {region_name}: {score:.4f}')

summary_text = '\n'.join(summary_lines)
summary_file = out_dir / 'executive_summary.txt'
summary_file.write_text(summary_text)
print('Saved executive summary:', summary_file)

# D) zip artifacts
artifact_candidates = [
    out_dir / 'proxy_labels_raw.csv',
    out_dir / 'proxy_labels_temporal.csv',
    out_dir / 'proxy_labels_severity.csv',
    out_dir / 'severity_bin_distribution_by_region.csv',
    out_dir / 'severity_bin_distribution_global.csv',
    out_dir / 'severity_region_summary.csv',
    out_dir / 'handcheck_subset.csv',
    out_dir / 'handcheck_template_to_fill.csv',
    out_dir / 'manual_validation_per_bin.csv',
    out_dir / 'handcheck_with_agreement.csv',
    out_dir / 'executive_summary.txt',
    out_dir / 'figures' / 'severity_bins_by_region_stacked.png',
    out_dir / 'figures' / 'severity_bins_global.png',
    out_dir / 'figures' / 'severity_score_histogram.png',
    out_dir / 'figures' / 'handcheck_contact_sheet.png',
]
existing_artifacts = [p for p in artifact_candidates if p.exists()]

zip_path = out_dir / 'final_submission_artifacts.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in existing_artifacts:
        zf.write(p, arcname=p.relative_to(out_dir))

print(f'Packaged {len(existing_artifacts)} artifacts -> {zip_path}')
print('Included files:')
for p in existing_artifacts:
    print('-', p.relative_to(out_dir))

In [ ]:
# make a contact sheet so manual labeling is faster

handcheck_csv = out_dir / 'handcheck_template_to_fill.csv'
if not handcheck_csv.exists():
    handcheck_csv = out_dir / 'handcheck_subset.csv'
if not handcheck_csv.exists():
    raise RuntimeError('No handcheck CSV found. Run hand-check generation cell first.')

df_contact = pd.read_csv(handcheck_csv).copy()
if len(df_contact) == 0:
    raise RuntimeError('Handcheck CSV is empty.')

# layout settings
n_cols = 6
max_items = min(len(df_contact), 120)
df_contact = df_contact.head(max_items).reset_index(drop=True)

n_rows = int(np.ceil(len(df_contact) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.2 * n_cols, 3.2 * n_rows))
axes = np.array(axes).reshape(-1)

for i, ax in enumerate(axes):
    if i >= len(df_contact):
        ax.axis('off')
        continue

    row = df_contact.iloc[i]
    mask_path = row.get('temporal_mask_path', None)
    pred_bin = row.get('severity_bin', np.nan)

    if isinstance(mask_path, str) and Path(mask_path).exists():
        mask = np.load(mask_path)
        ax.imshow(mask, cmap='Blues')
    else:
        ax.text(0.5, 0.5, 'Mask not found', ha='center', va='center')

    manual_bin = row.get('manual_severity_bin', np.nan)
    manual_txt = 'NA' if pd.isna(manual_bin) else str(int(manual_bin))
    ax.set_title(f"idx={i} | pred={int(pred_bin)} | manual={manual_txt}", fontsize=9)
    ax.axis('off')

plt.suptitle('Handcheck Contact Sheet (use idx to fill CSV rows)', y=1.002, fontsize=14)
plt.tight_layout()

contact_path = out_dir / 'figures' / 'handcheck_contact_sheet.png'
contact_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(contact_path, dpi=200, bbox_inches='tight')
plt.show()

print('Saved contact sheet:', contact_path)
print('Rows shown:', len(df_contact))
print('Use `idx` in the image to fill matching row in handcheck_template_to_fill.csv')